<a href="https://colab.research.google.com/github/Prktriesitout/HRAssistant/blob/main/HRscreening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-core

import os, sqlite3, hashlib, shutil
from datetime import date, timedelta
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print(os.listdir("."))   # you should see hr_tools.py and hr_assistant.db here

['.config', 'hr_assistant.db', 'hr_tools.py', 'sample_data']


In [2]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("hugging")
os.environ["HF_TOKEN"] = os.environ["HUGGINGFACEHUB_API_TOKEN"]

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=150, temperature=0.1, do_sample=True, return_full_text=False,
)
print("model ready")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model ready


In [3]:
import importlib, hr_tools
importlib.reload(hr_tools)

TEST_DB = "hr_test.db"
shutil.copy("hr_assistant.db", TEST_DB)

hr_tools.init(tokenizer, pipe, db_path=TEST_DB)
hr_tools.PULSE_SALT = "PUT_THE_SALT_FROM_seed.py_HERE"   # exact string from SALT in seed.py
hr_tools.TODAY = date(2026, 11, 16)                      # matches the seeded data's "today"

# helpers for the tests
def ask(uid, msg):
    print(f"\n{'='*70}\n[{uid}] {msg}\n{'-'*70}")
    print(hr_tools.run_agent(msg, uid))

def last_ticket(emp, kind=None):
    con = sqlite3.connect(TEST_DB)
    q = "select ticket_id from tickets where emp_id=?" + (" and kind=?" if kind else "")
    q += " order by cast(substr(ticket_id,4) as integer) desc limit 1"
    r = con.execute(q, (emp, kind) if kind else (emp,)).fetchone()
    con.close()
    return r[0] if r else None

def non_voter(survey):
    """An active employee who has NOT yet answered this survey."""
    con = sqlite3.connect(TEST_DB)
    hashes = {r[0] for r in con.execute("select participant_hash from pulse_participants where survey_id=?", (survey,))}
    emps = [r[0] for r in con.execute("select emp_id from employees where role='employee' and status='active'")]
    con.close()
    for e in emps:
        if hashlib.sha256((hr_tools.PULSE_SALT + e).encode()).hexdigest() not in hashes:
            return e

today = hr_tools.today()
next_monday = today + timedelta(days=(7 - today.weekday()) % 7 or 7)
print("today:", today, "| next Monday:", next_monday)

today: 2026-11-16 | next Monday: 2026-11-23


In [4]:
con = sqlite3.connect(TEST_DB)
for t in ["employees", "leave_balances", "tickets", "lifecycle_tasks", "pulse_responses", "pulse_participants", "escalations"]:
    print(f"{t:20s}", con.execute(f"select count(*) from {t}").fetchone()[0])

# schema markers: these confirm the DB was built from the ORIGINAL schema
for t in ("escalations", "tickets", "lifecycle_tasks"):
    row = con.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name=?", (t,)).fetchone()
    if not row:
        print(f"\n{t}: TABLE NOT FOUND")
        continue

    norm_sql = " ".join(row[0].split())  # Collapse multiple spaces/newlines into single spaces
    has_pk = "PRIMARY KEY (emp_id, task_id)" in norm_sql
    has_not_null = "emp_id TEXT NOT NULL" in norm_sql
    has_check = "kind = 'leave'" in norm_sql

    print(f"\n{t}: composite key={has_pk} | emp_id NOT NULL={has_not_null} | per-kind CHECK={has_check}")


# salt check: how many seeded participants can we recognise with the salt you typed?
hashes = {r[0] for r in con.execute("select participant_hash from pulse_participants where survey_id='Q4-2026'")}
emps = [r[0] for r in con.execute("select emp_id from employees")]
matched = [e for e in emps if hashlib.sha256((hr_tools.PULSE_SALT + e).encode()).hexdigest() in hashes]
print(f"\nQ4-2026 participants: {len(hashes)} | matched with your salt: {len(matched)}  (these should be equal)")
con.close()

# one tool called directly, no model involved
tools = {t.name: t for t in hr_tools.build_tools_for("E101")}
print("\nE101 balance:", tools["get_leave_balance"].invoke({}))

employees            60
leave_balances       180
tickets              120
lifecycle_tasks      35
pulse_responses      54
pulse_participants   54
escalations          4

escalations: composite key=False | emp_id NOT NULL=False | per-kind CHECK=False

tickets: composite key=False | emp_id NOT NULL=True | per-kind CHECK=False

lifecycle_tasks: composite key=True | emp_id NOT NULL=True | per-kind CHECK=False

Q4-2026 participants: 24 | matched with your salt: 0  (these should be equal)

E101 balance: {"casual": {"balance": 8, "pending_requests": 0, "available": 8}, "earned": {"balance": 15, "pending_requests": 0, "available": 15}, "sick": {"balance": 6, "pending_requests": 0, "available": 6}}


In [5]:
ask("E101", "How many casual leaves do I have left?")
ask("E101", f"Apply casual leave from {next_monday} to {next_monday + timedelta(days=1)} for a family function.")
ask("E101", "Apply earned leave for 40 days starting next month.")          # should fail: insufficient balance
ask("E101", "I want to claim INR 2,400 for a client lunch, meals category.")
ask("E101", "Claim INR 50,000 travel reimbursement.")                       # should fail: over the limit
ask("E101", "Show my tickets.")
ask("E101", "What are my pending onboarding tasks?")
ask("E101", "I finished the IT setup task, mark it done.")
ask("E102", f"What is the status of {last_ticket('E101', 'leave')}?")      # RBAC: Priya must NOT see Aarav's ticket


[E101] How many casual leaves do I have left?
----------------------------------------------------------------------
🔧 [E101] get_leave_balance({})
You currently have 8 casual leaves available to apply for.

[E101] Apply casual leave from 2026-11-23 to 2026-11-24 for a family function.
----------------------------------------------------------------------
🔧 [E101] apply_leave({'leave_type': 'casual', 'start_date': '2026-11-23', 'end_date': '2026-11-24', 'reason': 'family function'})
🔧 [E101] apply_leave({'leave_type': 'casual', 'start_date': '2026-11-23', 'end_date': '2026-11-24', 'reason': 'family function'})
🔧 [E101] apply_leave({'leave_type': 'casual', 'start_date': '2026-11-23', 'end_date': '2026-11-24', 'reason': 'family function'})
It seems there was an issue applying your leave request. Could you please try again? If the problem persists, we might need to escalate this to HR for further assistance.

[E101] Apply earned leave for 40 days starting next month.
--------------------

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔧 [E101] submit_reimbursement({'category': 'travel', 'amount': 50000, 'description': 'Business trip expenses'})
It seems there was an issue with the request. It appears that the system does not support specifying a specific daily limit for travel expenses. Could you please provide more details about your business trip so I can proceed with the claim?

[E101] Show my tickets.
----------------------------------------------------------------------
🔧 [E101] list_my_tickets({})
🔧 [E101] list_my_tickets({})
🔧 [E101] list_my_tickets({})
It seems there was an issue retrieving your tickets. Could you please try again later? If the problem persists, we should escalate this to HR for further assistance.

[E101] What are my pending onboarding tasks?
----------------------------------------------------------------------
🔧 [E101] get_onboarding_tasks({'pending_only': True})
🔧 [E101] get_onboarding_tasks({'pending_only': False})
🔧 [E101] get_onboarding_tasks({'pending_only': True})
🔧 [E101] get_onboa

In [6]:
tid = last_ticket("E101", "leave")
ask("E200", "Show pending requests from my team.")
ask("E200", f"Approve {tid} with comment 'Enjoy'.")
ask("E101", "How many casual leaves do I have left?")                       # should have dropped by 2
ask("E200", """Write appraisal feedback for E101. Notes: delivered the churn dashboard 2 weeks early,
good with stakeholders, SQL is strong, misses standups sometimes, docs are messy, wants to move into ML.""")
ask("E200", "Write appraisal feedback for E300. Notes: good work.")         # not a direct report: should fail


[E200] Show pending requests from my team.
----------------------------------------------------------------------
🔧 [E200] get_team_pending_requests({})
It seems there was an issue retrieving the pending requests. Let's try this again or check if the data is available later.

[E200] Approve None with comment 'Enjoy'.
----------------------------------------------------------------------
🔧 [E200] decide_request({'ticket_id': 'None', 'decision': 'approved', 'comment': 'Enjoy'})
It seems that there are no pending tickets for approval from your direct reports. Please check if you have any leave or reimbursement requests that need your attention.

[E101] How many casual leaves do I have left?
----------------------------------------------------------------------
🔧 [E101] get_leave_balance({})
You currently have 8 casual leaves available for application.

[E200] Write appraisal feedback for E101. Notes: delivered the churn dashboard 2 weeks early,
good with stakeholders, SQL is strong, miss

In [7]:
ask("E300", "Start offboarding for E102, last working day 2026-12-15, and show the checklist.")
ask("E300", "Draft an experience letter for E102.")
ask("E300", "Draft a welcome email for E101.")
ask("E200", "Start offboarding for E101, last working day 2026-12-15.")     # manager has no such tool: should refuse


[E300] Start offboarding for E102, last working day 2026-12-15, and show the checklist.
----------------------------------------------------------------------
🔧 [E300] start_offboarding({'emp_id': 'E102', 'last_working_day': '2026-12-15'})
There was an issue processing your request. It seems there was an internal error while trying to start the offboarding process for employee E102. Please try again later or contact HR support directly.

[E300] Draft an experience letter for E102.
----------------------------------------------------------------------
🔧 [E300] draft_hr_letter({'letter_type': 'experience_letter', 'emp_id': 'E102'})
🔧 [E300] draft_hr_letter({'emp_id': 'E102', 'letter_type': 'experience_letter'})
🔧 [E300] draft_hr_letter({'emp_id': 'E102', 'letter_type': 'experience_letter'})
🔧 [E300] draft_hr_letter({'emp_id': 'E102', 'letter_type': 'experience_letter'})
🔧 [E300] draft_hr_letter({'letter_type': 'experience_letter', 'emp_id': 'E102'})
Stopped: too many steps.

[E300] Draf

In [8]:
voter = non_voter("Q4-2026")
print("using employee who hasn't voted yet:", voter)

ask(voter, "Submit anonymous feedback for survey Q4-2026: the on-call rotation is exhausting, ask Priya about it.")
ask(voter, "Submit anonymous feedback for survey Q4-2026: trying a second time.")   # should say already responded
ask("E200", "Summarize pulse survey Q4-2026.")
ask("E101", "Summarize pulse survey Q4-2026.")                                      # employee has no such tool

# anonymity check: nothing in the response table links back to a person
con = sqlite3.connect(TEST_DB)
print("\nlatest stored response:", con.execute("select response_text, submitted_week from pulse_responses order by response_id desc limit 1").fetchone())
print("pulse_responses columns:", [c[1] for c in con.execute("pragma table_info(pulse_responses)")])
con.close()

using employee who hasn't voted yet: E102

[E102] Submit anonymous feedback for survey Q4-2026: the on-call rotation is exhausting, ask Priya about it.
----------------------------------------------------------------------
🔧 [E102] submit_pulse_feedback({'survey_id': 'Q4-2026', 'feedback': 'the on-call rotation is exhausting, ask Priya about it.'})
Your feedback has been submitted anonymously for the survey Q4-2026. The message "the on-call rotation is exhausting, ask Priya about it" has been recorded. Thank you for your input!

[E102] Submit anonymous feedback for survey Q4-2026: trying a second time.
----------------------------------------------------------------------
🔧 [E102] submit_pulse_feedback({'survey_id': 'Q4-2026', 'feedback': 'trying a second time'})
Your feedback has been submitted anonymously for the survey Q4-2026. Thank you for your input!

[E200] Summarize pulse survey Q4-2026.
----------------------------------------------------------------------
🔧 [E200] summarize_p

In [9]:
ask("E101", "An employee wants to report harassment by a colleague.")   # handled by the keyword guard, no LLM
ask("E101", "I am thinking of taking legal action against the company.")
ask("E101", "Apply casual leave next Monday for a family function.")    # normal request must NOT be escalated

con = sqlite3.connect(TEST_DB)
print("\nescalations:", con.execute("select escalation_id, emp_id, category, status from escalations order by escalation_id desc limit 3").fetchall())
print("\nlast audit rows:")
for r in con.execute("select actor_emp_id, tool_name, outcome from audit_log order by log_id desc limit 12"):
    print("  ", r)
print("\npulse submissions in audit log (must be 0):",
      con.execute("select count(*) from audit_log where tool_name='submit_pulse_feedback'").fetchone()[0])
con.close()


[E101] An employee wants to report harassment by a colleague.
----------------------------------------------------------------------


IntegrityError: NOT NULL constraint failed: escalations.status